<a href="https://colab.research.google.com/github/ricardocasallas3-ux/mesa-servicio-ia/blob/main/cargar_en_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cargar la base anonimizada en Google Colab

Procesa `incidentes_anonimizado.db` (SQLite) desde Google Drive con SQL y pandas.

El notebook **busca solo** el archivo en tu Drive; si no lo encuentra, lo **descarga por su ID**. No tienes que escribir rutas a mano.

Ejecuta *Entorno de ejecución → Ejecutar todo*.

## 1. Conectar Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Localizar la base de datos
Primero la busca en tu Drive; si no aparece, la descarga por su ID de archivo.

In [ ]:
import glob, os

FILE_ID = '1FZhxpcYeKsriLIHpKSJlfZNjygXZXqmX'   # id del archivo .db compartido
RUTA_DB = None

candidatos = glob.glob('/content/drive/MyDrive/**/incidentes_anonimizado.db', recursive=True)
if candidatos:
    RUTA_DB = candidatos[0]
    print('Encontrado en Drive:', RUTA_DB)
else:
    import gdown
    RUTA_DB = '/content/incidentes_anonimizado.db'
    gdown.download(id=FILE_ID, output=RUTA_DB, quiet=False)
    print('Descargado a:', RUTA_DB)

## 3. Abrir la base y listar las tablas

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(RUTA_DB)
tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)
print('Tablas disponibles:')
tablas

## 4. Cargar una tabla en un DataFrame
Pon en `TABLA` uno de los nombres que aparecieron arriba.

In [ ]:
TABLA = 'clasificaciones'   # <-- AJUSTAR segun la lista anterior
df = pd.read_sql('SELECT * FROM "%s"' % TABLA, con)
print('Filas, columnas:', df.shape)
df.head()

## 5. Exploracion inicial

In [ ]:
df.info()
df.describe(include='all')

## 6. Consultas SQL directas (opcional)

In [ ]:
# Ejemplo: conteo por clasificacion (ajusta el nombre de tabla/columna a los tuyos)
try:
    consulta = '''
    SELECT clasificacion, COUNT(*) AS cantidad
    FROM clasificaciones
    GROUP BY clasificacion
    ORDER BY cantidad DESC
    LIMIT 20
    '''
    display(pd.read_sql(consulta, con))
except Exception as e:
    print('Ajusta la consulta a tus tablas/columnas. Detalle:', e)

## 7. Alternativa: leer un CSV anonimizado (si lo subiste a Drive)

In [ ]:
csvs = glob.glob('/content/drive/MyDrive/**/*_anonimizado.csv', recursive=True)
print('CSV encontrados:', csvs)
if csvs:
    df_csv = pd.read_csv(csvs[0])
    display(df_csv.head())

## 8. Siguiente paso
Con el DataFrame cargado continua con limpieza, visualizaciones y analisis (ver el notebook del MVP y el de ACA 2).